# Select low-extendedness Lantern candidates

Previous name `lantern_low_extendedness_candidates_v2`.

Selection:

- `second_pass_score > 0.8`
- reject targets with `gaia_pm_sig_max > 3`
- reject targets with `gaia_parallax_sig_max > 3`
- compute the median `lsst_diaSource_extendedness` from retained candidate alerts
- keep `extendedness_median < 0.5` or targets with no finite extendedness values

The final output is a list of `target_id` values.


In [ ]:
from pathlib import Path
from collections.abc import Mapping
import json

import numpy as np
import pandas as pd


CANDIDATES_FILE = Path(
    "candidate_analysis_hybrid_multi_loci_v8_no_gaia_color_cut/"
    "second_pass_candidates_with_flags.parquet"
)

CANDIDATE_ALERTS_FILE = Path(
    "second_pass_output_hybrid_multi_loci_v3/"
    "second_pass_candidate_alerts.parquet"
)

OUTPUT_DIR = Path(
    "candidate_low_extendedness"
)

SECOND_PASS_CUT = 0.8
PM_SIG_CUT = 3.0
PARALLAX_SIG_CUT = 3.0
EXTENDEDNESS_CUT = 0.5

# Keep the same retained-alert date range used elsewhere.
MJD_MIN = 61187.0


In [ ]:
candidates = pd.read_parquet(
    CANDIDATES_FILE
)

candidate_alerts = pd.read_parquet(
    CANDIDATE_ALERTS_FILE
)

required_candidate_columns = {
    "target_id",
    "second_pass_score",
    "gaia_pm_sig_max",
    "gaia_parallax_sig_max",
}

missing = (
    required_candidate_columns
    - set(candidates.columns)
)

if missing:
    raise KeyError(
        f"Missing candidate columns: {sorted(missing)}"
    )

required_alert_columns = {
    "target_id",
    "mjd",
    "alert_properties",
}

missing = (
    required_alert_columns
    - set(candidate_alerts.columns)
)

if missing:
    raise KeyError(
        f"Missing alert columns: {sorted(missing)}"
    )

print(
    f"Candidate rows: {len(candidates):,}"
)

print(
    f"Candidate-alert rows: {len(candidate_alerts):,}"
)


In [ ]:
def parse_alert_properties(value):

    if isinstance(value, Mapping):
        return value

    if isinstance(value, str):
        try:
            parsed = json.loads(
                value
            )

            if isinstance(
                parsed,
                Mapping,
            ):
                return parsed

        except Exception:
            pass

    return {}


def to_float(value):

    try:
        return float(
            value
        )

    except (
        TypeError,
        ValueError,
    ):
        return np.nan


In [ ]:
alerts = candidate_alerts.copy()

alerts["mjd"] = pd.to_numeric(
    alerts["mjd"],
    errors="coerce",
)

alerts = alerts[
    alerts["mjd"] >= MJD_MIN
].copy()

props = alerts[
    "alert_properties"
].map(
    parse_alert_properties
)

alerts[
    "diaSource_extendedness"
] = props.map(
    lambda p: to_float(
        p.get(
            "lsst_diaSource_extendedness",
            np.nan,
        )
    )
)

extendedness_summary = (
    alerts
    .groupby(
        "target_id",
        as_index=False,
    )
    .agg(
        extendedness_median=(
            "diaSource_extendedness",
            "median",
        ),
        extendedness_n=(
            "diaSource_extendedness",
            "count",
        ),
    )
)

print(
    f"Targets with alert rows after MJD cut: "
    f"{len(extendedness_summary):,}"
)


In [ ]:
analysis = candidates.merge(
    extendedness_summary,
    on="target_id",
    how="left",
    validate="one_to_one",
)

analysis[
    "second_pass_keep"
] = (
    analysis[
        "second_pass_score"
    ] > SECOND_PASS_CUT
)

analysis[
    "gaia_pm_keep"
] = ~(
    analysis[
        "gaia_pm_sig_max"
    ] > PM_SIG_CUT
)

analysis[
    "gaia_parallax_keep"
] = ~(
    analysis[
        "gaia_parallax_sig_max"
    ] > PARALLAX_SIG_CUT
)

analysis[
    "extendedness_keep"
] = (
    analysis[
        "extendedness_median"
    ].isna()
    | (
        analysis[
            "extendedness_median"
        ] < EXTENDEDNESS_CUT
    )
)

selected = analysis[
    analysis[
        "second_pass_keep"
    ]
    & analysis[
        "gaia_pm_keep"
    ]
    & analysis[
        "gaia_parallax_keep"
    ]
    & analysis[
        "extendedness_keep"
    ]
].copy()

selected = (
    selected
    .sort_values(
        "second_pass_score",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)

print(
    f"Second-Pass > {SECOND_PASS_CUT}: "
    f"{analysis['second_pass_keep'].sum():,}"
)

print(
    f"Pass Gaia PM cut: "
    f"{analysis['gaia_pm_keep'].sum():,}"
)

print(
    f"Pass Gaia parallax cut: "
    f"{analysis['gaia_parallax_keep'].sum():,}"
)

print(
    f"Median extendedness < {EXTENDEDNESS_CUT} or NaN: "
    f"{analysis['extendedness_keep'].sum():,}"
)

print(
    f"Final selected targets: "
    f"{len(selected):,}"
)


In [ ]:
target_ids = (
    selected[
        "target_id"
    ]
    .astype(str)
    .tolist()
)

target_ids[:20]


In [ ]:
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

txt_file = (
    OUTPUT_DIR
    / "low_extendedness_target_ids.txt"
)

csv_file = (
    OUTPUT_DIR
    / "low_extendedness_target_ids.csv"
)

txt_file.write_text(
    "\n".join(
        target_ids
    )
    + (
        "\n"
        if target_ids
        else ""
    )
)

pd.DataFrame(
    {
        "target_id": target_ids
    }
).to_csv(
    csv_file,
    index=False,
)

print(
    f"Saved {len(target_ids):,} target IDs"
)

print(
    f"  {txt_file}"
)

print(
    f"  {csv_file}"
)
